In [1]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime, timedelta
import talib
from functools import reduce
from PIL import Image
import plotly.io as pio

In [5]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess
from utils.constants import CURRENCY_MAPPER, PLOTLY_CURRENCY_NORMALIZER

In [6]:
CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE', 'SGLLV']
}
CODES_TO_COUNTRY = {v : k for k,vv in CODES.items() for v in vv}
FEATURES = ['high','low','close', 'adx', 'plus_di', 'minus_di', 'adx_trend']
DAYS = 365
PERIOD = 16 # currently work best for 16 days
ADX_THRESHOLD = 25

In [7]:
# Get 1 year data
end_date = datetime.today().date()
start_date = end_date - timedelta(days = DAYS)
world_engine = SQLModule.get_engine(country = 'world')
# Get all exchange rate
query = f"""
    SELECT
        *
    FROM daily_average_exchange_rate_usd_based
    WHERE
        date >= DATE '{start_date}'
        AND
        date <= DATE '{end_date}'
    ORDER BY date
"""
ex_rate = pd.read_sql_query(query, world_engine)
ex_rate.set_index('date', inplace = True)
# fill nan for each rate
for col in ex_rate.columns:
    # Backfilling the variables
    ex_rate[col] = ex_rate[col].fillna(method = 'bfill').fillna(method = 'ffill')

In [8]:
dfs = []
for stock_code,country in CODES_TO_COUNTRY.items():
    engine = SQLModule.get_engine(country = country)
    stock_query = f"""
        SELECT
            date,
            high,
            low,
            close
        FROM transaction
        WHERE
            stock_code = '{stock_code}'
            AND
            date >= DATE '{start_date}'
            AND
            date <= DATE '{end_date}'
        ORDER BY date
    """
    df = pd.read_sql_query(stock_query, engine)
    # Remove nan value
    df.index = df['date']
    df = df.drop('date', axis = 1)
    df = StockPriceProcess.remove_invalid_data(df, country = country)

    # convert to usd
    if country != 'united_states':
        df = df.join(ex_rate[[CURRENCY_MAPPER[country]]])
        for price_type in ['high', 'low', 'close']:
            df[price_type] = df[price_type] / df[CURRENCY_MAPPER[country]]

    df['adx'] = talib.ADX(df['high'], df['low'], df['close'], timeperiod = PERIOD).to_numpy()
    df['minus_di'] = talib.MINUS_DI(df['high'], df['low'], df['close'], timeperiod = PERIOD).to_numpy()
    df['plus_di'] = talib.PLUS_DI(df['high'], df['low'], df['close'], timeperiod = PERIOD).to_numpy()
    df['adx_trend'] = 'wait'
    for i in range(1,len(df)):
        if df.iloc[i - 1]['adx'] < ADX_THRESHOLD and df.iloc[i]['adx'] > ADX_THRESHOLD and df.iloc[i]['plus_di'] > df.iloc[i]['minus_di']:
            df['adx_trend'].iloc[i] = 'bull'
        elif df.iloc[i - 1]['adx'] < ADX_THRESHOLD and df.iloc[i]['adx'] > ADX_THRESHOLD and df.iloc[i]['plus_di'] < df.iloc[i]['minus_di']:
            df['adx_trend'].iloc[i] = 'bear'

    df = df[FEATURES].iloc[PERIOD * 2 - 1:]

    # Change column to multi-index
    df.columns = pd.MultiIndex.from_tuples([(stock_code,col) for col in df.columns])

    dfs.append(df.reset_index())

result = reduce(lambda l,r: pd.merge(l,r, on='date', how='outer'), dfs).sort_values(by='date').reset_index(drop = True)
result.index = result['date']
df = result.drop('date', axis = 1)
df.iloc[-5:,:]

ACB                                                       \
                high       low     close        adx    plus_di   minus_di   
date                                                                        
2025-02-12  1.018410  1.004700  1.004700  16.609668  26.821986  17.860345   
2025-02-13  1.007828  1.000000  1.005871  16.450039  25.811466  19.449730   
2025-02-14  1.020047  1.010220  1.012186  16.954259  29.908034  18.130227   
2025-02-17  1.022862  1.014978  1.016949  17.563959  29.724792  17.193241   
2025-02-18  1.026451  1.016581  1.016581  18.309615  30.046822  16.359521   

                           TPG                      ...        TNE             \
           adx_trend      high       low     close  ...    plus_di   minus_di   
date                                                ...                         
2025-02-12      wait  2.802743  2.739760  2.752357  ...  28.745200  17.395460   
2025-02-13      wait  2.745665  2.707967  2.707967  ...  27.058851  20.040801   
2025-02-14      wait  2.779445  2.722592  2.773128  ...  29.157651  18.773886   
2025-02-17      wait  2.803221  2.758726  2.784152  ...  31.346908  17.656904   
2025-02-18      wait  2.818586  2.786774  2.812224  ...  30.143320  17.153073   

                         SGLLV                                            \
           adx_trend      high       low     close        adx    plus_di   
date                                                                       
2025-02-12      wait  6.701391  6.487249  6.531337  17.227915  23.075328   
2025-02-13      wait  6.534305  6.408646  6.502891  16.170005  22.251190   
2025-02-14      wait  6.569596  6.405357  6.500110  15.287288  22.214977   
2025-02-17      wait  6.706120  6.470929  6.706120  14.998996  24.635651   
2025-02-18      wait  6.756972  6.648809  6.680622  14.917497  25.353475   

                                 
             minus_di adx_trend  
date                             
2025-02-12  20.898003      wait  
2025-02-13  22.385706      wait  
2025-02-14  21.323937      wait  
2025-02-17  19.883409      wait  
2025-02-18  19.245628      wait  

[5 rows x 28 columns]

In [9]:
pio.renderers.default = "plotly_mimetype"
for num,(stock_code,country) in enumerate(CODES_TO_COUNTRY.items()):
    # Get data from that stock code
    _df = df[[(stock_code, col) for col in FEATURES]].droplevel(0, axis = 1)
    # merge with ex_rate
    _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
    _df['close'] = _df['close'] * _df[CURRENCY_MAPPER[country]]
    fig = make_subplots(rows = 2, cols = 1)
    # Plot the price
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['close'] / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'blue'),
            name = 'close_price'
        ),
        row = 1, col = 1
    )
    fig.update_yaxes(
        title = 'Close price', 
        tickprefix = f'{CURRENCY_MAPPER[CODES_TO_COUNTRY[stock_code]]} ',
        ticksuffix = f' * {PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]]}',
        showgrid = True, 
        gridcolor = 'gray', 
        tickfont=dict(color='blue'), 
        row = 1, col = 1
    )

    # Then, plot the RSI
    for metric, color in zip(['adx', 'plus_di', 'minus_di'], ['purple', 'green', 'red']):
        fig.add_trace(
            go.Scatter(
                x = _df.index, 
                y = _df[metric],  
                marker = dict(color = color),
                opacity = 0.5 if metric != 'adx' else 1.0,
                name = metric,
            ),
            row = 2, col = 1
        )

    # Plot the signals
    for date, row in _df.iterrows():
        price = row['close']
        signal = row['adx_trend']
        if signal == 'bull':
            fig.add_layout_image(
                dict(
                    source = Image.open("img/bull_icon.png"),
                    x = date,  # x location (date)
                    xanchor = "center",
                    yanchor = 'top',
                    y=price / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  # y location (price)
                    xref="x",  # Referencing the x-axis
                    yref="y",  # Referencing the y-axis
                    sizex = 5 * 24 * 60 * 60 * 1000,  # Image width
                    sizey = 5,    # Image height
                    sizing="contain",
                    opacity = 1.0,
                    layer="above"
                ),
                row = 1, col = 1
            )
        elif signal == 'bear':
            fig.add_layout_image(
                dict(
                    source = Image.open("img/bear_icon.png"),
                    x = date,  # x location (date)
                    xanchor = "center",
                    yanchor = 'bottom',
                    y=price / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],   # y location (price)
                    xref="x",  # Referencing the x-axis
                    yref="y",  # Referencing the y-axis
                    sizex = 5 * 24 * 60 * 60 * 1000,  # Image width
                    sizey = 5,    # Image height
                    sizing="contain",
                    opacity = 1.0,
                    layer="above"
                ),
                row = 1, col = 1
            )
    fig.update_yaxes(title = 'Values', showgrid = True, gridcolor = 'gray', tickfont=dict(color='black'), row = 2, col = 1)
    fig.update_xaxes(title = 'Date', showgrid = False)
    fig.update_layout(
        plot_bgcolor = 'white' if num % 2 == 0 else "rgb(210, 210, 210)",
        paper_bgcolor = 'white' if num % 2 == 0 else "rgb(210, 210, 210)",
        height = 800,
        font = dict(size = 20),
        title = f'[{country}] {stock_code}'
    )
    fig.show()